# **Introduction to RAG**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [Foundations: From IR to RAG](#Foundations:-From-IR-to-RAG)
  - [Introduction](#Introduction)
  - [The Evolution from IR to Modern AI](#The-Evolution-from-IR-to-Modern-AI)
  - [What is Retrieval-Augmented Generation (RAG)?](#What-is-Retrieval-Augmented-Generation-(RAG)?)
  - [Key Benefits and Real-World Applications](#Key-Benefits-and-Real-World-Applications)
  - [Conclusion and Next Steps](#Conclusion-and-Next-Steps)
- [RAG in Action: A Simple Workflow](#RAG-in-Action:-A-Simple-Workflow)
  - [Introduction](#Introduction)
  - [The RAG Workflow: Four Key Steps](#The-RAG-Workflow:-Four-Key-Steps)
  - [Indexing: Organizing Documents](#Indexing:-Organizing-Documents)
  - [Retrieval: Locating Relevant Information](#Retrieval:-Locating-Relevant-Information)
  - [Query Augmentation: Creating Context-Rich Prompts](#Query-Augmentation:-Creating-Context-Rich-Prompts)
  - [Generation: Producing Tailored Answers](#Generation:-Producing-Tailored-Answers)
  - [Conclusion and Next Steps](#Conclusion-and-Next-Steps)
- [Fighting Hallucinations: RAG's Role in Trustworthy LLMs](#Fighting-Hallucinations:-RAG's-Role-in-Trustworthy-LLMs)
  - [Introduction](#Introduction)
  - [RAG: Grounding LLMs in a Knowledge Base](#RAG:-Grounding-LLMs-in-a-Knowledge-Base)
  - [A Financial Knowledge Base](#A-Financial-Knowledge-Base)
  - [Output Analysis: Accuracy Showdown](#Output-Analysis:-Accuracy-Showdown)
  - [Final Code](#Final-Code)
  - [Handle Missing Data in RAG](#Handle-Missing-Data-in-RAG)
  - [Conclusion and Next Steps](#Conclusion-and-Next-Steps)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Foundations: From IR to RAG

## Introduction

Hello there, welcome to the first lesson of the **"Introduction to RAG"** course! I'm glad you're joining me on this journey into *Retrieval-Augmented Generation* (RAG). RAG sits at the intersection of two powerful areas in AI: traditional **information retrieval** (IR), which powers search engines, and modern **Generative AI** (GenAI), which creates human-like text. Think of it as giving AI a "fact-checking assistant" before it answers your questions.

By the end of this course, you'll see how RAG combines the best of both worlds: the precision of search engines and the creativity of language models. Let's get started already!

## The Evolution from IR to Modern AI

**Phase 1: Classic IR Systems**

Early systems (like library databases or early Google) focused on keyword matching. You'd type "climate change effects," and get a list of articles. But you had to read them all to find the answers you were looking for.

**Phase 2: Generative AI Breakthroughs**

With advances in NLP, LLMs like GPT-3 moved beyond just retrieving documents to generating relevant, fluent responses. But they had weaknesses: **factual reliability** (*hallucinations*, where models make up facts they don't know about) and **static knowledge** (e.g., an LLM trained on data up to 2021 can't know 2023 medical guidelines). Imagine asking, "What's the latest policy for treating this particular disease?" and getting a plausible-sounding, but outdated or wrong answer.

**Phase 3: RAG Bridges the Gap**

RAG solves this by first retrieving up-to-date or domain-specific data from a designated source (e.g., latest guidelines, internal documents, ...). It's like a journalist writing a story: instead of relying solely on their memory and intuition, they first gather verified facts and quotes from reliable sources, ensuring the article is accurate and grounded in reality.

To see more concretely how RAG differs from other AI workflows, let's compare Classic IR, GenAI, and RAG in a simplified step-by-step pipeline:

* **Classic IR**: Query → Processing → Lookup → Matching → Ranking → Results
* **GenAI**: Data → Preprocessing → Training → Fine-tuning → Generation (Inference) → Evaluation
* **RAG**: Query → Retrieval → Processing → Fusion → Generation → Output

RAG stands out because of the “Retrieval” step prior to “Generation.” This means the model isn't relying solely on internal parameters—it's explicitly pulling relevant, up-to-date data. The “Fusion” step then merges what's retrieved with the model's existing semantic understanding, helping ensure factual and context-specific responses.

## What is Retrieval-Augmented Generation (RAG)?

In simple terms, RAG uses IR techniques to fetch the most relevant data first and then feeds that data into a generative model. As the model crafts its response, it draws on these retrieved sources, which **ground** its output in reality (reducing the risk of making facts up).

For instance, if you're building an AI assistant for a big company's knowledge base, you don't want the assistant to rely solely on a large language model's internal knowledge. Instead, you want it to pull up the latest policy documents, guidelines or internal codebases and base its response on this specific information. That's the core of RAG: retrieve, then generate.

Another example: a legal research assistant using RAG would first retrieve clauses from relevant case law, and then generate a summary that cites specific precedents. Without RAG, the model might invent a fake court ruling!

## Key Benefits and Real-World Applications

What are some of the key benefits of adopting RAG-enabled workflows?

* **Factual Guardrails**: Retrieval acts as a "safety net" against hallucinations, making RAG systems less likely to invent or mix up information.

* **Dynamic Knowledge**: Update answers by updating the data source — no need to retrain the LLM! (Note: the retrieval system must be periodically refreshed with new data - RAG is not a "set and forget" technique.)

* **Domain-Specific Expertise**: RAG can be tailored using specialized databases or documents, such as medical records or legal documents.

Real-world use cases include:

* **Healthcare**: 🏥 Chatbots that pull latest drug trial data before advising patients.

* **E-commerce**: 🛒 Product assistants that reference real-time inventory and specs.

* **Education**: 📚 Tutors that generate explanations using approved textbooks.

## Conclusion and Next Steps

We've explored the transition from classic information retrieval to the modern world of AI generation, and how RAG unites these approaches. Moving forward in this learning path, you'll dig deeper into core components like text embeddings and specialized vector databases. That's where you'll gain hands-on experience, practice critical skills, and see the full power of RAG in action. I'm excited to continue this journey with you as we unlock even more ways to build effective, reliable AI solutions!

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# RAG in Action: A Simple Workflow

## Introduction

Welcome to our second lesson in the “**Introduction to RAG**” course! In the previous lesson, you learned how RAG evolved from traditional Information Retrieval (IR). Today, we'll connect those ideas to concrete code and illustrate a simple end-to-end RAG workflow. By the end of this lesson, you'll see how indexing, retrieval, prompt augmentation, and final text generation fit together to produce a targeted answer.

In this lesson, we showcase a scenario involving “Project Chimera.” Think of this as an internal project of a company — here, it's just an example to demonstrate how a naive, context-free system can invent inaccurate details, whereas a RAG-based system will provide reliable answers using information from an authoritative knowledge base. Please note that we are deliberately using extremely simplified methods (like simple keyword matching) to illustrate how each part of a RAG pipeline can work. Later in this course, we will explore more realistic and robust approaches—such as using embeddings and vector databases—for each component of our RAG pipeline.

## The RAG Workflow: Four Key Steps

Before diving into the code, let's take a quick look at the four primary steps of our simple RAG workflow:

* **Indexing**: Documents are structured in a way that makes them easy to search.

* **Retrieval**: The most relevant piece of text is fetched based on a user query.

* **Prompt (Query) Augmentation**: The retrieved text is combined with the user's question to form a context-rich prompt.

* **Generation**: A language model processes the prompt and produces a final answer anchored to the provided text.

This process ensures that answers are backed by your data, reducing the risk of fabricated or off-topic responses. Let's examine each step through code!

## Indexing: Organizing Documents

We start by defining our knowledge base. Below, “Project Chimera” serves as the example domain:

In [ ]:
KNOWLEDGE_BASE = {
    "doc1": {
        "title": "Project Chimera Overview",
        "content": (
            "Project Chimera is a research initiative focused on developing "
            "novel bio-integrated interfaces. It aims to merge biological "
            "systems with advanced computing technologies."
        )
    },
    "doc2": {
        "title": "Chimera's Neural Interface",
        "content": (
            "The core component of Project Chimera is a neural interface "
            "that allows for bidirectional communication between the brain "
            "and external devices. This interface uses biocompatible "
            "nanomaterials."
        )
    },
    "doc3": {
        "title": "Applications of Chimera",
        "content": (
            "Potential applications of Project Chimera include advanced "
            "prosthetics, treatment of neurological disorders, and enhanced "
            "human-computer interaction. Ethical considerations are paramount."
        )
    }
}

Here's a quick breakdown:

* We define a Python dictionary named `KNOWLEDGE_BASE` that contains multiple documents.

* Each entry has an ID (e.g., `"doc1"`) and both a `title` and `content` field.
* “Project Chimera” information is now the authoritative data source for the RAG system.

Keep in mind this is a very simplified approach for educational purposes; in a real-world production scenario, your `KNOWLEDGE_BASE` would consist of more advanced components, such as a *Vector Database*. But don't worry, we'll be dealing with these databases in Course 3!

## Retrieval: Locating Relevant Information

Next, we create a function to return the best document from our knowledge base, based on simple keyword overlap:

In [ ]:
def rag_retrieval(query, documents):
    query_words = set(query.lower().split())
    best_doc_id = None
    best_overlap = 0

    for doc_id, doc in documents.items():
        # Compare the query words with the document's content words
        doc_words = set(doc['content'].lower().split())
        overlap = len(query_words.intersection(doc_words))

        if overlap > best_overlap:
            best_overlap = overlap
            best_doc_id = doc_id

    # Return the best document, or None if nothing matched
    return documents.get(best_doc_id)

Let's walk through the code:
* The query is split into lowercase words and stored in a set.
* Each document's text is similarly tokenized.
* The function picks the document with the greatest word overlap.
* If no match is found, it returns `None`.

## Query Augmentation: Creating Context-Rich Prompts

Once we retrieve the relevant document, we augment the user's original question with that document's content. This additional context significantly reduces hallucinations because the language model sees both the question and real data.

In [ ]:
def rag_generation(query, document):
    if document:
        snippet = f"{document['title']: {document['content']}}"
        prompt = f"Using the following information: '{snippet}', answer: {query}"
    else:
        prompt = f"No relevant information found. Answer directly: {query}"

Here's the process in detail:

* If a document was found, we include its `title` and main `content` as a `snippet` in the prompt.

* We pass the combined query and document text to the language model.

* If no document matches, we still form a direct prompt.

Given our `KNOWLEDGE_BASE` and the query `"What is the main goal of Project Chimera?"`, the resulting RAG-augmented prompt may be similar to the following:

*Using the following information: 'Chimera's Neural Interface: The core component of Project Chimera is a neural interface that allows for bidirectional communication between the brain and external devices. This interface uses biocompatible nanomaterials.', answer: What is the main goal of Project Chimera?*

## Generation: Producing Tailored Answers

Finally, let's see the difference between a naive approach—one that might invent details if it doesn't have any context—and a RAG approach, which leverages the knowledge base.

In [ ]:
def get_llm_response(prompt):
    """
    This function interfaces with a language model to generate a response based on the provided prompt.

    Parameters:
    - prompt (str): A string containing the question or task for the language model, potentially augmented with additional context.

    Returns:
    - response (str): The generated text from the language model, which aims to answer the question or fulfill the task described in the prompt.
    """
    pass

def naive_generation(query):
    # This approach ignores the knowledge base
    prompt = f"Answer directly the following query: {query}"
    return get_llm_response(prompt)

def rag_generation(query, document):
    if document:
        snippet = f"{document['title']: {document['content']}}"
        prompt = f"Using the following information: '{snippet}', answer {query}"
    else:
        prompt = f"No relevant information found. Answer directly: {query}"
    return get_llm_response(prompt)

query = "What is the main goal of Project Chimera?"

naive_answer = naive_generation(query)
print(f'Naive approach: {naive_answer}')

doc = rag_retrieval(query, KNOWLEDGE_BASE)
rag_answer = rag_generation(query, doc)
print(f'RAG approach: {rag_answer}')

Let's break down this section:

* `naive_generation(query)` can easily lead to random or inaccurate answers regarding “Project Chimera”, such as: `"The main goal of Project Chimera is to develop advanced artificial intelligence systems that can enhance human capabilities and improve decision-making processes across various fields."`.

* `rag_generation(query, doc)` provides contextual information from the knowledge base, ensuring the answer is grounded: `"The main goal of Project Chimera is to enable bidirectional communication between the brain and external devices through the use of a neural interface."`.

Seeing both approaches (and their actual output) helps you compare how naive answers can deviate from your authoritative data, while RAG-based responses stay closer to truth.

## Conclusion and Next Steps

You've now implemented:

* A simple knowledge base indexing scheme.

* Basic retrieval to find the most relevant document.

* Prompt augmentation to combine user queries and reference data.

* Generation that relies on actual context, lowering the chance of hallucinations.

Up next, you'll get hands-on practice with these steps in coding exercises. As you progress, you'll see how RAG can be extended to tackle more complex tasks and domains—whether you're talking about a literal “Project Chimera” or a real-world internal project.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Fighting Hallucinations: RAG's Role in Trustworthy LLMs

## Introduction

Welcome to the final lesson of the "**Introduction to RAG**" course! In the previous lessons, you've learned the fundamentals of RAG, including how to build a basic RAG pipeline. Now, we'll address a critical issue in Large Language Models (LLMs): **hallucinations**.

Hallucinations occur when an LLM generates plausible-sounding but factually incorrect information. This can be a major problem in real-world applications where accuracy is paramount. Imagine relying on an LLM for medical advice or financial analysis, only to receive fabricated information! In this lesson, we'll explore how RAG can significantly reduce hallucinations by grounding the LLM in a reliable knowledge base.

## RAG: Grounding LLMs in a Knowledge Base

As you learned in the previous lesson, **RAG** addresses the problem of hallucinations by providing the LLM with relevant context retrieved from a knowledge base. This "grounding" helps the LLM generate more accurate and reliable responses.

To recap, the basic RAG pipeline involves these steps:

1. **Indexing**: Organizing documents into a knowledge base.

2. **Retrieval**: Identifying the most relevant document(s) from the knowledge base based on the user's query.

3. **Augmentation**: Combining the retrieved document(s) with the original query to create a context-rich prompt.

4. **Generation**: Feeding the augmented prompt to the LLM to generate a response.


## A Financial Knowledge Base

Let's define our `KNOWLEDGE_BASE` for this lesson. This is a dictionary containing information about stock prices for AAPL, MSFT, and TSLA on April 13th and 14th, 2023.

In [ ]:
KNOWLEDGE_BASE = {
    "AAPL": {
        "title": "AAPL Stock (April 2023)",
        "content": (
            "On 2023-04-13, AAPL opened at $160.50, closed at $162.30, with a high of $163.00 and a low of $159.90. "
            "Trading volume was 80 million shares. "
            "On 2023-04-14, AAPL opened at $161.10, closed at $162.80, with a high of $163.50 and a low of $160.50. "
            "Trading volume was 85 million shares."
        )
    },
    "MSFT": {
        "title": "MSFT Stock (April 2023)",
        "content": (
            "On 2023-04-13, MSFT opened at $285.00, closed at $288.50, with a high of $290.00 and a low of $283.50. "
            "Trading volume was 35 million shares. "
            "On 2023-04-14, MSFT opened at $286.00, closed at $289.00, with a high of $291.50 and a low of $284.70. "
            "Trading volume was 40 million shares."
        )
    },
    "TSLA": {
        "title": "TSLA Stock (April 2023)",
        "content": (
            "On 2023-04-13, TSLA opened at $185.00, closed at $187.00, with a high of $189.00 and a low of $184.50. "
            "Trading volume was 50 million shares. "
            "On 2023-04-14, TSLA opened at $186.00, closed at $188.50, with a high of $190.00 and a low of $185.50. "
            "Trading volume was 55 million shares."
        )
    }
}

This `KNOWLEDGE_BASE` is a dictionary where keys are stock symbols (`AAPL`, `MSFT`, `TSLA`) and values are dictionaries containing the title and content of a document related to that stock. The content includes the opening price, closing price, highest and lowest prices, and trading volume for April 13th and 14th, 2023.

The rest of the codebase is the same as the one in the previous lesson, so we won't discuss that again here.

## Output Analysis: Accuracy Showdown

Now, let's compare the outputs of the `naive_generation` and `rag_generation` functions when answering the same query.

Here's the query we'll use:

In [ ]:
query = (
    "Write a short summary of the stock market performance on April 14, "
    "2023 for the following symbols: AAPL, MSFT, TSLA.\n"
    "Your summary should include:\n"
    "For each symbol:\n"
    "- The opening price\n"
    "- The closing price\n"
    "- The highest and lowest prices of the day\n"
    "- The trading volume"
)

This query asks for a summary of the stock market performance on April 14, 2023, for `AAPL`, `MSFT`, and `TSLA`, including specific details like opening price, closing price, and trading volume.

When we run the `naive_generation` function with this query, the LLM generates a response based solely on its pre-trained knowledge. The output might look something like this:

```markdown
Naive approach:
    On April 14, 2023, the stock market performance for the specified symbols was as follows:

    **AAPL (Apple Inc.)**
    - Opening Price: $167.50
    - Closing Price: $169.00
    - Highest Price: $170.00
    - Lowest Price: $166.80
    - Trading Volume: 55 million shares

    **MSFT (Microsoft Corporation)**
    - Opening Price: $290.00
    - Closing Price: $292.50
    - Highest Price: $293.00
    - Lowest Price: $289.50
    - Trading Volume: 30 million shares

    **TSLA (Tesla, Inc.)**
    - Opening Price: $185.00
    - Closing Price: $188.50
    - Highest Price: $189.00
    - Lowest Price: $184.50
    - Trading Volume: 45 million shares
```

Notice that the LLM provides plausible-sounding numbers for all the requested information. However, these numbers are absolutely incorrect because the LLM is hallucinating. It's making up the data!

Now, let's run the `rag_generation` function with the same query. The output might look something like this:

```markdown
RAG approach:
    ### Stock Market Performance Summary for April 14, 2023

    **AAPL (Apple Inc.)**
    - **Opening Price:** $161.10
    - **Closing Price:** $162.80
    - **Highest Price:** $163.50
    - **Lowest Price:** $160.50
    - **Trading Volume:** 85 million shares

    **MSFT (Microsoft Corp.)**
    - **Opening Price:** Data not provided.
    - **Closing Price:** Data not provided.
    - **Highest Price:** Data not provided.
    - **Lowest Price:** Data not provided.
    - **Trading Volume:** Data not provided.

    **TSLA (Tesla Inc.)**
    - **Opening Price:** Data not provided.
    - **Closing Price:** Data not provided.
    - **Highest Price:** Data not provided.
    - **Lowest Price:** Data not provided.
    - **Trading Volume:** Data not provided.

    *Note: Information for MSFT and TSLA is not available in the provided data.*
```

In this case, the RAG approach provides accurate information for `AAPL`, as it's grounded in the `KNOWLEDGE_BASE`. However, it states that it doesn't have information for `MSFT` and `TSLA`; how can that be, given that our `KNOWLEDGE_BASE` includes information about those symbols as well?

If you remember the `rag_retrieval` function from the previous lesson, you might have already guessed why.

## Final Code

In [ ]:
from scripts.llm import get_llm_response

KNOWLEDGE_BASE = {
    "AAPL": {
        "title": "AAPL Stock (April 2023)",
        "content": (
            "On 2023-04-13, AAPL opened at $160.50, closed at $162.30, with a high of $163.00 and a low of $159.90. "
            "Trading volume was 80 million shares. "
            "On 2023-04-14, AAPL opened at $161.10, closed at $162.80, with a high of $163.50 and a low of $160.50. "
            "Trading volume was 85 million shares."
        )
    },
    "MSFT": {
        "title": "MSFT Stock (April 2023)",
        "content": (
            "On 2023-04-13, MSFT opened at $285.00, closed at $288.50, with a high of $290.00 and a low of $283.50. "
            "Trading volume was 35 million shares. "
            "On 2023-04-14, MSFT opened at $286.00, closed at $289.00, with a high of $291.50 and a low of $284.70. "
            "Trading volume was 40 million shares."
        )
    },
    "TSLA": {
        "title": "TSLA Stock (April 2023)",
        "content": (
            "On 2023-04-13, TSLA opened at $185.00, closed at $187.00, with a high of $189.00 and a low of $184.50. "
            "Trading volume was 50 million shares. "
            "On 2023-04-14, TSLA opened at $186.00, closed at $188.50, with a high of $190.00 and a low of $185.50. "
            "Trading volume was 55 million shares."
        )
    }
}


def naive_generation(query):
    prompt = f"Answer directly the following query: {query}"
    return get_llm_response(prompt)


def rag_retrieval(query, knowledge_base):
    query_words = set(query.lower().split())
    relevant_documents = []
    for doc_id, doc in knowledge_base.items():
        doc_words = set(doc['title'].lower().split() + doc['content'].lower().split())
        overlap = len(query_words.intersection(doc_words))
        if overlap > 0:
            relevant_documents.append(doc)
    return relevant_documents


def rag_generation(query, documents):
    if documents:
        snippets = "\n".join([f"{doc['title']}: {doc['content']}" for doc in documents])
        prompt = f"Using the following information: '{snippets}', answer: {query}"
    else:
        prompt = f"No relevant information found. Answer directly: {query}"
    return get_llm_response(prompt)


if __name__ == "__main__":
    query = (
        "Write a short summary of the stock market performance on April 14, "
        "2023 for the following symbols: AAPL, MSFT, TSLA.\n"
        "Your summary should include:\n"
        "For each symbol:\n"
        "- The opening price\n"
        "- The closing price\n"
        "- The highest and lowest prices of the day\n"
        "- The trading volume"
    )
    # Naive approach hallucinates (generates a random plausible, but incorrect, answer)
    print("Naive approach:\n", naive_generation(query))
    # RAG approach prevents hallucination by "grounding" the answer (providing additional context)
    retrieved_docs = rag_retrieval(query, KNOWLEDGE_BASE)
    print("\n\nRAG approach:\n", rag_generation(query, retrieved_docs))

Output:

```markdown
Naive approach:
 On April 14, 2023, the stock market performance for the specified symbols was as follows:

**AAPL (Apple Inc.)**
- Opening Price: $167.50
- Closing Price: $169.00
- Highest Price: $170.00
- Lowest Price: $166.80
- Trading Volume: 55 million shares

**MSFT (Microsoft Corporation)**
- Opening Price: $290.00
- Closing Price: $292.50
- Highest Price: $293.00
- Lowest Price: $289.50
- Trading Volume: 25 million shares

**TSLA (Tesla, Inc.)**
- Opening Price: $185.00
- Closing Price: $188.50
- Highest Price: $189.00
- Lowest Price: $184.50
- Trading Volume: 40 million shares

Please note that the prices and volumes are illustrative and may not reflect actual market data.


RAG approach:
 On April 14, 2023, the stock market performance for AAPL, MSFT, and TSLA was as follows:

**AAPL (Apple Inc.):**
- Opening Price: $161.10
- Closing Price: $162.80
- Highest Price: $163.50
- Lowest Price: $160.50
- Trading Volume: 85 million shares

**MSFT (Microsoft Corporation):**
- Opening Price: $286.00
- Closing Price: $289.00
- Highest Price: $291.50
- Lowest Price: $284.70
- Trading Volume: 40 million shares

**TSLA (Tesla, Inc.):**
- Opening Price: $186.00
- Closing Price: $188.50
- Highest Price: $190.00
- Lowest Price: $185.50
- Trading Volume: 55 million shares

Overall, all three stocks showed positive closing prices on this day, with AAPL and TSLA experiencing moderate trading volumes, while MSFT had a slightly lower volume.
```

## Handle Missing Data in RAG

In this task, you'll modify the RAG pipeline to address missing or insufficient data. Specifically, you must update the prompt in the rag_generation function so that:

1. If the retrieved document fully covers the query, the model confidently provides an answer.

2. If any critical piece of information is missing or incomplete, the model should politely refuse to answer, indicating there isn't enough data to respond accurately.

By strictly relying on the retrieved context, you'll help prevent hallucinations and ensure your RAG system only responds when it is certain about the information provided.

In [ ]:
from scripts.llm import get_llm_response

KNOWLEDGE_BASE = {
    "AAPL": {
        "title": "AAPL Stock (April 2023)",
        "content": (
            "On 2023-04-13, AAPL opened at $160.50, closed at $162.30, with a high of $163.00 and a low of $159.90. "
            "Trading volume was 80 million shares. "
            "On 2023-04-14, AAPL opened at $161.10, closed at $162.80, with a high of $163.50 and a low of $160.50. "
            "Trading volume was 85 million shares."
        )
    },
    "MSFT": {
        "title": "MSFT Stock (April 2023)",
        "content": (
            "On 2023-04-13, MSFT opened at $285.00, closed at $288.50, with a high of $290.00 and a low of $283.50. "
            "Trading volume was 35 million shares. "
            "On 2023-04-14, MSFT opened at $286.00, closed at $289.00, with a high of $291.50 and a low of $284.70. "
            "Trading volume was 40 million shares."
        )
    },
    "TSLA": {
        "title": "TSLA Stock (April 2023)",
        "content": (
            "On 2023-04-13, TSLA opened at $185.00, closed at $187.00, with a high of $189.00 and a low of $184.50. "
            "Trading volume was 50 million shares. "
            "On 2023-04-14, TSLA opened at $186.00, closed at $188.50, with a high of $190.00 and a low of $185.50. "
            "Trading volume was 55 million shares."
        )
    }
}


def naive_generation(query):
    prompt = f"Answer directly the following query: {query}"
    return get_llm_response(prompt)


def rag_retrieval(query, knowledge_base):
    query_words = set(query.lower().split())
    best_doc_id = None
    best_overlap = 0
    for doc_id, doc in knowledge_base.items():
        doc_words = set(doc["content"].lower().split())
        overlap = len(query_words.intersection(doc_words))
        if overlap > best_overlap:
            best_overlap = overlap
            best_doc_id = doc_id
    return knowledge_base.get(best_doc_id)


def rag_generation(query, document):
    # TODO: Check if 'document' is valid and includes the necessary data.
    #       If it does, construct a prompt that instructs the model to answer
    #       ONLY if the relevant information is present in 'document'.
    #       Otherwise, politely refuse to provide an answer.
    if document:
        snippet = f"{document['title']}: {document['content']}"
        prompt = f"Using the following information: '{snippet}', answer query: {query}"
        prompt += f"\n"
        prompt += f"Answer ONLY if the relevant information is present in the provided information given in context. Otherwise, politely refuse to provide an answer."
    else:
        prompt = f"No relevant information found. Answer directly: {query}"
    return get_llm_response(prompt)


if __name__ == "__main__":
    query = (
        "Write a short summary of the stock market performance on April 14, "
        "2023 for the following symbols: NVDA, GOOG.\n"
        "Your summary should include:\n"
        "For each symbol:\n"
        "- The opening price\n"
        "- The closing price\n"
        "- The highest and lowest prices of the day\n"
        "- The trading volume"
    )
    print("Naive approach:\n", naive_generation(query))
    retrieved_doc = rag_retrieval(query, KNOWLEDGE_BASE)
    print("\n\nRAG approach:\n", rag_generation(query, retrieved_doc))

Output:

```markdown
Naive approach:
 On April 14, 2023, the stock market performance for the specified symbols was as follows:

**NVDA (NVIDIA Corporation):**
- Opening Price: $270.00
- Closing Price: $275.50
- Highest Price of the Day: $276.00
- Lowest Price of the Day: $268.50
- Trading Volume: 25 million shares

**GOOG (Alphabet Inc.):**
- Opening Price: $105.00
- Closing Price: $107.25
- Highest Price of the Day: $107.50
- Lowest Price of the Day: $104.75
- Trading Volume: 15 million shares


RAG approach:
 I'm sorry, but the provided information only includes details about AAPL stock performance on April 13 and April 14, 2023. There is no information available for NVDA or GOOG.
```

## Conclusion and Next Steps

Congratulations on completing the "**Introduction to RAG**" course! You've learned the core principles of RAG, from its evolution from information retrieval to building a basic RAG pipeline. In this final lesson, you've seen how RAG can significantly improve the accuracy and reliability of LLM responses by grounding them in a knowledge base.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[CodeSignal: Introduction to RAG Course](https://codesignal.com/learn/courses/introduction-to-rag)**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)